# Assignment 3 - Exercise 5
This notebook solves Exercise 5 (points 1-6) using `trajectories.pickle`.


## 5.1 Load trajectories and MC evaluation for $v_\pi(s)$
We use **first-visit Monte Carlo** with $\gamma=1.0$ on trajectories generated by a fixed policy $\pi$.


In [ ]:
from pathlib import Path
import pickle
import numpy as np
from collections import defaultdict, Counter

pickle_path = Path('trajectories.pickle')
with pickle_path.open('rb') as f:
    trajectories = pickle.load(f)

print(f'Loaded {len(trajectories)} trajectories from {pickle_path}')
print(f'Example trajectory length: {len(trajectories[0])}')
print('Example step:', trajectories[0][0])



In [ ]:
def mc_first_visit_v_q(trajectories, gamma=1.0):
    # First-visit Monte Carlo estimates for V_pi(s) and Q_pi(s,a).
    # Assumes each step is (s, a, r, s_next), where s=(x,y).
    returns_v = defaultdict(list)
    returns_q = defaultdict(list)

    for traj in trajectories:
        G = 0.0
        visited_s = set()
        visited_sa = set()

        for s, a, r, s_next in reversed(traj):
            s = (int(s[0]), int(s[1]))
            a = int(a)
            r = float(r)
            G = r + gamma * G

            if s not in visited_s:
                returns_v[s].append(G)
                visited_s.add(s)

            if (s, a) not in visited_sa:
                returns_q[(s, a)].append(G)
                visited_sa.add((s, a))

    V = {s: float(np.mean(gs)) for s, gs in returns_v.items()}
    Q = {(s, a): float(np.mean(gs)) for (s, a), gs in returns_q.items()}
    return V, Q, returns_v, returns_q

V, Q, returns_v, returns_q = mc_first_visit_v_q(trajectories, gamma=1.0)
print(f'Estimated V for {len(V)} states')
print(f'Estimated Q for {len(Q)} state-action pairs')



## 5.2 MC evaluation for $q_\pi(a,s)$
The same function above computes $Q_\pi(s,a)$ from first-visit returns.
Below we inspect a few entries.


In [ ]:
for i, ((s, a), q) in enumerate(Q.items()):
    if i == 10:
        break
    print(f's={s}, a={a}, Q={q:.3f}, n={len(returns_q[(s,a)])}')



## 5.3 Plot $v_\pi(s)$
State space is visualized as a `31 x 100` grid with `(x,y)` coordinates.


In [ ]:
value_grid = np.full((31, 100), np.nan, dtype=float)
for (x, y), v in V.items():
    if 0 <= x < 31 and 0 <= y < 100:
        value_grid[x, y] = v

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 4.8))
    im = plt.imshow(value_grid, origin='lower', aspect='auto', cmap='viridis')
    plt.colorbar(im, label='V_pi(s)')
    plt.title('Exercise 5.3: Monte Carlo State-Value Function')
    plt.xlabel('y')
    plt.ylabel('x')
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print('matplotlib not installed in this runtime; run this cell in your notebook environment to see the plot.')



## 5.4 Can we perform MC improvement?
Yes, but only **partially** on this fixed dataset:

$$\pi_{new}(s) = \arg\max_{a \in A_{obs}(s)} Q_\pi(s,a)$$

This is greedy over observed actions only, not guaranteed globally optimal.


In [ ]:
state_action_count = Counter()
for traj in trajectories:
    for s, a, r, s_next in traj:
        s = (int(s[0]), int(s[1]))
        a = int(a)
        state_action_count[(s, a)] += 1

actions_by_state = defaultdict(set)
for (s, a) in Q:
    actions_by_state[s].add(a)

greedy_policy_observed = {
    s: max(actions, key=lambda a: Q[(s, a)])
    for s, actions in actions_by_state.items()
}

behavior_policy_empirical = {}
tmp = defaultdict(Counter)
for (s, a), c in state_action_count.items():
    tmp[s][a] = c
for s, cnt in tmp.items():
    behavior_policy_empirical[s] = cnt.most_common(1)[0][0]

changed_states = sum(
    1 for s in greedy_policy_observed
    if greedy_policy_observed[s] != behavior_policy_empirical.get(s)
)

print(f'States with >=2 observed actions: {sum(len(v) >= 2 for v in actions_by_state.values())}/{len(actions_by_state)}')
print(f'States where greedy(Q) differs from empirical behavior action: {changed_states}/{len(greedy_policy_observed)}')



## 5.5 Are all trajectories equally useful?
No. They are not equally useful for MC evaluation/improvement.

- Frequently visited states/actions reduce variance and give reliable estimates.
- Rare `(s,a)` pairs have few samples and noisy returns.
- Improvement quality depends on coverage of state-action space.


In [ ]:
sample_counts = np.array([len(v) for v in returns_q.values()])
print(f'First-visit samples per (s,a): mean={sample_counts.mean():.2f}, median={np.median(sample_counts):.1f}, min={sample_counts.min()}, max={sample_counts.max()}')
print(f'Fraction with <=3 samples: {(sample_counts <= 3).mean():.3f}')



## 5.6 Can we perform MC control with only fixed trajectories?
Not full MC control.

With a fixed offline dataset, we cannot keep sampling new episodes from updated policies, so we cannot run the usual iterative MC control loop with policy improvement/evaluation until convergence.

What we can do: limited offline policy improvement on covered state-action pairs.
What we cannot claim: guaranteed optimal policy over the full MDP.


## 5.7 Choose a starting policy (for `mountain/GridWorld-v1`)
Starting policy: **forward-biased centerline policy**.

- Action meanings in v1: `0=forward`, `1=left`, `2=forward-left`, `3=right`, `4=forward-right`.
- Goal is mostly progress in `y` while keeping row `x` near center (`x≈15`).
- Policy:
  - if `x < 14`: use `forward-right` (move down toward center while progressing)
  - if `x > 16`: use `forward-left` (move up toward center while progressing)
  - else: use `forward`

This is a reasonable non-random prior that still reaches many states and gives MC control a better start than pure random behavior.


In [ ]:
import numpy as np

def heuristic_start_action(state):
    x, y = state
    if x < 14:
        return 4  # forward-right
    if x > 16:
        return 2  # forward-left
    return 0      # forward



## 5.8 Perform MC control on `mountain/GridWorld-v1`
On-policy first-visit MC control with epsilon-greedy improvement.


In [ ]:
import gymnasium as gym

# Ensure package is importable (requires `pip install -e mountain` from Assignment 3 directory)
try:
    import mountain
except Exception as e:
    print('Could not import mountain package. Run: pip install -e mountain')
    raise

env = gym.make('mountain/GridWorld-v1')
print('Action space size:', env.action_space.n)



In [ ]:
from collections import defaultdict

def obs_to_state(obs):
    pos = obs['agent']['pos']
    return (int(pos[0]), int(pos[1]))

def epsilon_greedy_action(Q, state, n_actions, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)

    q_vals = np.array([Q[(state, a)] for a in range(n_actions)], dtype=float)
    if np.allclose(q_vals, q_vals[0]):
        return heuristic_start_action(state)

    best = np.flatnonzero(q_vals == q_vals.max())
    return int(np.random.choice(best))

def generate_episode(env, Q, epsilon, max_steps=1000):
    obs, info = env.reset()
    state = obs_to_state(obs)

    episode = []  # (s, a, r)
    total_reward = 0.0

    for _ in range(max_steps):
        action = epsilon_greedy_action(Q, state, env.action_space.n, epsilon)
        next_obs, reward, terminated, truncated, info = env.step(action)
        episode.append((state, action, float(reward)))
        total_reward += float(reward)

        state = obs_to_state(next_obs)
        if terminated or truncated:
            break

    return episode, total_reward, state

def mc_control_on_policy(env, episodes=5000, gamma=1.0, epsilon=0.10):
    Q = defaultdict(float)
    returns = defaultdict(list)

    reward_history = []
    final_y_history = []

    for ep in range(1, episodes + 1):
        episode, ep_return, final_state = generate_episode(env, Q, epsilon)
        reward_history.append(ep_return)
        final_y_history.append(final_state[1])

        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = r + gamma * G
            if (s, a) not in visited:
                returns[(s, a)].append(G)
                Q[(s, a)] = float(np.mean(returns[(s, a)]))
                visited.add((s, a))

        if ep % 500 == 0:
            avg_r = float(np.mean(reward_history[-500:]))
            avg_y = float(np.mean(final_y_history[-500:]))
            print(f'Episode {ep:4d} | avg return(last 500)={avg_r:8.3f} | avg final y(last 500)={avg_y:6.2f}')

    return Q, reward_history, final_y_history

Q_control, reward_hist, final_y_hist = mc_control_on_policy(
    env,
    episodes=5000,
    gamma=1.0,
    epsilon=0.10,
)



In [ ]:
# Build greedy policy from learned Q

def greedy_policy_action(Q, state, n_actions):
    q_vals = np.array([Q[(state, a)] for a in range(n_actions)], dtype=float)
    best = np.flatnonzero(q_vals == q_vals.max())
    if len(best) == 0:
        return heuristic_start_action(state)
    return int(np.random.choice(best))

def evaluate_greedy_policy(env, Q, episodes=200):
    returns = []
    successes = 0
    finals = []

    for _ in range(episodes):
        obs, _ = env.reset()
        s = obs_to_state(obs)
        done = False
        ep_return = 0.0

        while not done:
            a = greedy_policy_action(Q, s, env.action_space.n)
            obs2, r, terminated, truncated, info = env.step(a)
            ep_return += float(r)
            s = obs_to_state(obs2)
            done = terminated or truncated

        returns.append(ep_return)
        finals.append(s[1])
        if s[1] >= 99:
            successes += 1

    return {
        'avg_return': float(np.mean(returns)),
        'success_rate': successes / episodes,
        'avg_final_y': float(np.mean(finals)),
    }

metrics = evaluate_greedy_policy(env, Q_control, episodes=200)
print(metrics)



In [ ]:
# Optional training curves
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(reward_hist, alpha=0.6)
    ax[0].set_title('Episode Return During MC Control')
    ax[0].set_xlabel('Episode')
    ax[0].set_ylabel('Return')

    ax[1].plot(final_y_hist, alpha=0.6)
    ax[1].set_title('Final y Position During MC Control')
    ax[1].set_xlabel('Episode')
    ax[1].set_ylabel('Final y')

    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print('matplotlib not installed in this runtime; run this cell in your notebook environment for plots.')



### Notes for write-up
- If success rate is high and average final `y` is near `99`, MC control is learning a strong policy.
- This does not prove global optimality, but it is strong empirical evidence.
- Tune `episodes` and `epsilon` if convergence is slow.


## 5.9 Did you manage to learn an optimal policy?
Use repeated train/eval runs to answer this empirically, and report challenges (runtime + sensitivity to exploration/starting behavior).


In [ ]:
import time

def run_mc_control_experiment(env, seeds=(0, 1, 2), episodes=3000, epsilon=0.10, eval_episodes=300):
    rows = []

    for seed in seeds:
        np.random.seed(seed)
        t0 = time.perf_counter()

        Q_seed, reward_seed, final_y_seed = mc_control_on_policy(
            env,
            episodes=episodes,
            gamma=1.0,
            epsilon=epsilon,
        )

        train_sec = time.perf_counter() - t0
        m = evaluate_greedy_policy(env, Q_seed, episodes=eval_episodes)

        rows.append({
            'seed': seed,
            'train_seconds': train_sec,
            'avg_return': m['avg_return'],
            'success_rate': m['success_rate'],
            'avg_final_y': m['avg_final_y'],
            'last500_train_return': float(np.mean(reward_seed[-500:])) if len(reward_seed) >= 500 else float(np.mean(reward_seed)),
            'last500_train_final_y': float(np.mean(final_y_seed[-500:])) if len(final_y_seed) >= 500 else float(np.mean(final_y_seed)),
        })

    return rows

rows = run_mc_control_experiment(
    env,
    seeds=(0, 1, 2),
    episodes=3000,
    epsilon=0.10,
    eval_episodes=300,
)

for r in rows:
    print(r)

mean_success = float(np.mean([r['success_rate'] for r in rows]))
mean_final_y = float(np.mean([r['avg_final_y'] for r in rows]))
mean_time = float(np.mean([r['train_seconds'] for r in rows]))
print('
Summary:')
print({'mean_success_rate': mean_success, 'mean_avg_final_y': mean_final_y, 'mean_train_seconds': mean_time})



### 5.9 How to conclude from your results
- If success rate is consistently high across seeds (e.g., near 1.0) and average final `y` is near `99`, you can claim the policy is **empirically near-optimal**.
- Do **not** claim a formal proof of global optimality from MC experiments alone.
- Challenges to report:
  1. **Computational time**: MC updates happen after full episodes, so many episodes are needed.
  2. **Exploration sensitivity**: too low `epsilon` can trap learning; too high `epsilon` slows exploitation.
  3. **Starting policy effect**: better initial behavior improves early data quality and convergence speed.
  4. **Return variance**: stochastic transitions increase variance, requiring more episodes for stable estimates.


## 5.10 MC control on `mountain/GridWorld-v2`
SpaceZ can move in 8 directions (`GridWorld-v2`). We attempt MC control and compare convergence and computational cost against v1.


In [32]:
# v2-specific starting policy (still goal-directed and center-seeking)
def heuristic_start_action_v2(state):
    x, y = state

    # Move toward center row while progressing forward in y when possible
    if x < 14:
        return 4  # forward-right
    if x > 16:
        return 2  # forward-left
    return 0      # forward


def epsilon_greedy_action_fallback(Q, state, n_actions, epsilon, fallback_policy):
    if np.random.rand() < epsilon:
        return np.random.randint(n_actions)

    q_vals = np.array([Q[(state, a)] for a in range(n_actions)], dtype=float)
    if np.allclose(q_vals, q_vals[0]):
        return fallback_policy(state)

    best = np.flatnonzero(q_vals == q_vals.max())
    return int(np.random.choice(best))


def generate_episode_generic(env, Q, epsilon, fallback_policy, max_steps=1200):
    obs, _ = env.reset()
    state = obs_to_state(obs)

    episode = []
    total_reward = 0.0

    for _ in range(max_steps):
        action = epsilon_greedy_action_fallback(Q, state, env.action_space.n, epsilon, fallback_policy)
        next_obs, reward, terminated, truncated, _ = env.step(action)

        episode.append((state, action, float(reward)))
        total_reward += float(reward)
        state = obs_to_state(next_obs)

        if terminated or truncated:
            break

    return episode, total_reward, state


def mc_control_on_policy_generic(env, episodes=3000, gamma=1.0, epsilon=0.10, fallback_policy=heuristic_start_action):
    Q = defaultdict(float)
    returns = defaultdict(list)

    reward_history = []
    final_y_history = []

    for ep in range(1, episodes + 1):
        episode, ep_return, final_state = generate_episode_generic(
            env, Q, epsilon, fallback_policy=fallback_policy
        )
        reward_history.append(ep_return)
        final_y_history.append(final_state[1])

        G = 0.0
        visited = set()
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = r + gamma * G
            if (s, a) not in visited:
                returns[(s, a)].append(G)
                Q[(s, a)] = float(np.mean(returns[(s, a)]))
                visited.add((s, a))

        if ep % 500 == 0:
            print(f'Episode {ep:4d} | avg return(last 500)={np.mean(reward_history[-500:]):8.3f} | avg final y(last 500)={np.mean(final_y_history[-500:]):6.2f}')

    return Q, reward_history, final_y_history



In [33]:
import time

env_v1 = gym.make('mountain/GridWorld-v1')
env_v2 = gym.make('mountain/GridWorld-v2')

# Train MC control on v2
t0 = time.perf_counter()
Q_v2, reward_v2, finaly_v2 = mc_control_on_policy_generic(
    env_v2,
    episodes=3000,
    gamma=1.0,
    epsilon=0.10,
    fallback_policy=heuristic_start_action_v2,
)
train_time_v2 = time.perf_counter() - t0

metrics_v2 = evaluate_greedy_policy(env_v2, Q_v2, episodes=300)
print('v2 metrics:', metrics_v2)
print(f'v2 train time (s): {train_time_v2:.2f}')



Episode  500 | avg return(last 500)=-173.589 | avg final y(last 500)=  5.01
Episode 1000 | avg return(last 500)=-175.160 | avg final y(last 500)=  3.53
Episode 1500 | avg return(last 500)=-175.411 | avg final y(last 500)=  3.28
Episode 2000 | avg return(last 500)=-175.314 | avg final y(last 500)=  3.34
Episode 2500 | avg return(last 500)=-175.961 | avg final y(last 500)=  2.70
Episode 3000 | avg return(last 500)=-175.470 | avg final y(last 500)=  3.18
v2 metrics: {'avg_return': -178.51484996123872, 'success_rate': 0.0, 'avg_final_y': 0.0}
v2 train time (s): 47.55


In [34]:
# Compare computational cost and final quality between v1 and v2 (same budget)
def train_eval_once(env_id, episodes=3000, epsilon=0.10, seed=0):
    np.random.seed(seed)
    env_local = gym.make(env_id)

    fallback = heuristic_start_action if env_local.action_space.n == 5 else heuristic_start_action_v2

    t0 = time.perf_counter()
    Q_local, r_hist, y_hist = mc_control_on_policy_generic(
        env_local,
        episodes=episodes,
        gamma=1.0,
        epsilon=epsilon,
        fallback_policy=fallback,
    )
    train_s = time.perf_counter() - t0

    m = evaluate_greedy_policy(env_local, Q_local, episodes=300)

    return {
        'env': env_id,
        'train_seconds': train_s,
        'avg_return': m['avg_return'],
        'success_rate': m['success_rate'],
        'avg_final_y': m['avg_final_y'],
        'last500_train_return': float(np.mean(r_hist[-500:])) if len(r_hist) >= 500 else float(np.mean(r_hist)),
        'last500_train_final_y': float(np.mean(y_hist[-500:])) if len(y_hist) >= 500 else float(np.mean(y_hist)),
    }

res_v1 = train_eval_once('mountain/GridWorld-v1', episodes=3000, epsilon=0.10, seed=0)
res_v2 = train_eval_once('mountain/GridWorld-v2', episodes=3000, epsilon=0.10, seed=0)
print(res_v1)
print(res_v2)

print('Relative training-time ratio (v2 / v1):', res_v2['train_seconds'] / res_v1['train_seconds'])



Episode  500 | avg return(last 500)= -57.058 | avg final y(last 500)= 98.50
Episode 1000 | avg return(last 500)= -49.486 | avg final y(last 500)= 98.94
Episode 1500 | avg return(last 500)= -48.124 | avg final y(last 500)= 99.00
Episode 2000 | avg return(last 500)= -46.631 | avg final y(last 500)= 98.93
Episode 2500 | avg return(last 500)= -44.817 | avg final y(last 500)= 98.94
Episode 3000 | avg return(last 500)= -44.276 | avg final y(last 500)= 98.99
Episode  500 | avg return(last 500)=-171.912 | avg final y(last 500)=  6.67
Episode 1000 | avg return(last 500)=-174.095 | avg final y(last 500)=  4.56
Episode 1500 | avg return(last 500)=-174.429 | avg final y(last 500)=  4.25
Episode 2000 | avg return(last 500)=-176.385 | avg final y(last 500)=  2.26
Episode 2500 | avg return(last 500)=-176.383 | avg final y(last 500)=  2.28
Episode 3000 | avg return(last 500)=-176.026 | avg final y(last 500)=  2.63
{'env': 'mountain/GridWorld-v1', 'train_seconds': 77.57253381700139, 'avg_return': -39.5

### 5.10 Interpretation template
- If `success_rate` and `avg_final_y` are lower on v2 for the same episode budget, MC control is struggling more to converge in the larger action space.
- If `train_seconds` is higher for v2, report that extra movement options increase search space and computational cost.
- Typical conclusion: v2 usually needs more episodes (or better exploration strategy) than v1 to reach similar policy quality.
